# DSFB SRD Figure Regeneration

This notebook reads CSV outputs from `output-dsfb-srd/<timestamp>/` and regenerates the three paper-facing figures for the deterministic Structural Regime Dynamics demonstrator.

If the outputs are not already present in Colab, the setup cell can clone the DSFB repository, install Rust, generate a fresh SRD run, and then plot from that run.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

RUN_NAME = None  # Example: "20260312-214530"
REPO_ROOT = Path("/content/dsfb")
OUTPUT_ROOT = None  # Optional: Path("/content/output-dsfb-srd")
AUTO_CLONE_REPO = True
AUTO_GENERATE_OUTPUTS = True
DSFB_REPO_URL = "https://github.com/infinityabundance/dsfb.git"

def run_checked(command, cwd=None, env=None):
    print("+", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def unique_paths(paths):
    ordered = []
    seen = set()
    for candidate in paths:
        candidate = Path(candidate).expanduser()
        if candidate not in seen:
            seen.add(candidate)
            ordered.append(candidate)
    return ordered

def resolve_repo_root():
    candidates = []

    if REPO_ROOT is not None:
        candidates.append(Path(REPO_ROOT).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.append(Path("/content/dsfb"))

    for candidate in unique_paths(candidates):
        if (candidate / "crates/dsfb-srd/Cargo.toml").exists():
            return candidate

    return None

def ensure_repo_root():
    repo_root = resolve_repo_root()
    if repo_root is not None:
        return repo_root

    if not AUTO_CLONE_REPO:
        return None

    target = Path(REPO_ROOT).expanduser() if REPO_ROOT is not None else Path("/content/dsfb")
    if target.exists() and not (target / "crates/dsfb-srd/Cargo.toml").exists():
        raise FileExistsError(
            f"Cannot clone DSFB into {target}: path exists but is not the DSFB repository"
        )

    if not target.exists():
        run_checked(["git", "clone", "--depth", "1", DSFB_REPO_URL, str(target)])

    return target

def cargo_env():
    env = os.environ.copy()
    cargo_bin = str(Path.home() / ".cargo" / "bin")
    env["PATH"] = cargo_bin + os.pathsep + env.get("PATH", "")
    return env

def ensure_cargo():
    env = cargo_env()
    if shutil.which("cargo", path=env["PATH"]):
        return env

    run_checked(
        ["sh", "-c", "curl https://sh.rustup.rs -sSf | sh -s -- -y"],
        env=env,
    )

    if not shutil.which("cargo", path=env["PATH"]):
        raise RuntimeError("Rust installation completed but cargo is still unavailable")

    return env

def discover_output_root(repo_root=None):
    candidates = []

    if OUTPUT_ROOT is not None:
        candidates.append(Path(OUTPUT_ROOT).expanduser())

    if repo_root is not None:
        candidates.append(repo_root / "output-dsfb-srd")

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidates.append(base / "output-dsfb-srd")

    candidates.extend(
        [
            Path("/content/dsfb/output-dsfb-srd"),
            Path("/content/output-dsfb-srd"),
            Path("/content/drive/MyDrive/dsfb/output-dsfb-srd"),
            Path("/content/drive/MyDrive/output-dsfb-srd"),
        ]
    )

    unique_candidates = unique_paths(candidates)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate, unique_candidates

    return None, unique_candidates

def searched_message(paths):
    searched = "\n".join(f"- {candidate}" for candidate in paths)
    return (
        "Could not locate output-dsfb-srd.\n"
        "Set OUTPUT_ROOT directly, or clone/copy the repo or output folder into Colab first.\n"
        "Searched:\n"
        f"{searched}"
    )

def ensure_output_root():
    repo_root = ensure_repo_root()
    output_root, searched = discover_output_root(repo_root)
    if output_root is not None:
        return repo_root, output_root

    if not AUTO_GENERATE_OUTPUTS or repo_root is None:
        raise FileNotFoundError(searched_message(searched))

    env = ensure_cargo()
    run_checked(
        [
            "cargo",
            "run",
            "--manifest-path",
            "crates/dsfb-srd/Cargo.toml",
            "--release",
            "--bin",
            "dsfb-srd-generate",
        ],
        cwd=repo_root,
        env=env,
    )

    output_root, searched = discover_output_root(repo_root)
    if output_root is None:
        raise FileNotFoundError(searched_message(searched))

    return repo_root, output_root

REPO_ROOT, OUTPUT_ROOT = ensure_output_root()
available_runs = sorted(path for path in OUTPUT_ROOT.iterdir() if path.is_dir())
if not available_runs:
    raise FileNotFoundError(f"No timestamped runs found under {OUTPUT_ROOT}")

RUN_DIR = OUTPUT_ROOT / RUN_NAME if RUN_NAME else available_runs[-1]
if not RUN_DIR.exists():
    raise FileNotFoundError(f"Requested run directory not found: {RUN_DIR}")

print(f"Using output root: {OUTPUT_ROOT}")
print(f"Using run directory: {RUN_DIR}")

In [ ]:
manifest = pd.read_csv(RUN_DIR / "run_manifest.csv")
events = pd.read_csv(RUN_DIR / "events.csv")
threshold_sweep = pd.read_csv(RUN_DIR / "threshold_sweep.csv")
transition_sharpness = pd.read_csv(RUN_DIR / "transition_sharpness.csv")
time_local = pd.read_csv(RUN_DIR / "time_local_metrics.csv")

manifest

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for n_events, subset in threshold_sweep.groupby("n_events"):
    subset = subset.sort_values("tau_threshold")
    ax.plot(
        subset["tau_threshold"],
        subset["reachable_fraction"],
        linewidth=2,
        label=f"N={n_events}",
    )

ax.set_title("Figure 1: Connectivity Collapse $\\rho(\\tau)$")
ax.set_xlabel("Trust threshold $\\tau$")
ax.set_ylabel("Reachable fraction $\\rho(\\tau)$")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for n_events, subset in transition_sharpness.groupby("n_events"):
    subset = subset.sort_values("tau_midpoint")
    ax.plot(
        subset["tau_midpoint"],
        subset["abs_drho_dtau"],
        linewidth=2,
        label=f"N={n_events}",
    )

ax.set_title("Figure 2: Transition Sharpness $|d\\rho / d\\tau|$")
ax.set_xlabel("Threshold midpoint")
ax.set_ylabel("Absolute derivative")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
manifest_row = manifest.iloc[0]
shock_start = manifest_row["shock_start"]
shock_end = manifest_row["shock_end"]

time_local = time_local.copy()
time_local["window_midpoint"] = 0.5 * (time_local["window_start"] + time_local["window_end"])

fig, ax = plt.subplots(figsize=(10, 6))
for tau_threshold, subset in time_local.groupby("tau_threshold"):
    subset = subset.sort_values("window_midpoint")
    ax.plot(
        subset["window_midpoint"],
        subset["reachable_fraction"],
        linewidth=2,
        label=f"tau={tau_threshold:.2f}",
    )

ax.axvspan(shock_start, shock_end, color="tomato", alpha=0.18, label="shock interval")
ax.set_title("Figure 3: Time-Local Connectivity During Structural Regimes")
ax.set_xlabel("Event index")
ax.set_ylabel("Window reachable fraction")
ax.legend()
fig.tight_layout()
plt.show()